# Governed Tool Use with Deterministic Guardrails

This cookbook shows how to add deterministic governance guardrails around Claude's tool use — scanning tool arguments for PII, enforcing tool allowlists, capping session cost, and producing structured audit trails.

Unlike LLM-based content moderation (which uses Claude to judge its own output), this approach is fully deterministic: regex patterns + policy rules, no additional LLM call, under 2ms overhead per decision.

**What you'll learn:**
- Intercept tool calls before execution to enforce governance policies
- Detect PII (SSN, credit cards, emails) in tool arguments
- Restrict which tools the agent can call (allowlist)
- Enforce per-session cost budgets
- Produce structured audit evidence (TEEC receipts) for compliance

## Setup

In [ ]:
%pip install -q anthropic tealtiger

In [ ]:
import os
import anthropic

# Set your API key
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "your-key-here"))

## Define Tools

We'll define three tools: `search_database` (safe), `send_email` (safe), and `delete_records` (dangerous — should be blocked by governance).

In [ ]:
tools = [
    {
        "name": "search_database",
        "description": "Search the internal database for employee or financial records.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "The search query"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "send_email",
        "description": "Send an email to a recipient.",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"}
            },
            "required": ["to", "subject", "body"]
        }
    },
    {
        "name": "delete_records",
        "description": "Delete records from a database table.",
        "input_schema": {
            "type": "object",
            "properties": {
                "table": {"type": "string"},
                "condition": {"type": "string"}
            },
            "required": ["table", "condition"]
        }
    }
]

## Configure Governance Policies

[TealTiger](https://github.com/agentguard-ai/tealtiger) provides deterministic governance — all evaluation is regex + policy rules with no LLM in the path.

In [ ]:
from tealtiger import TealEngine, GovernancePolicy, GovernanceMode
import json

engine = TealEngine(
    mode=GovernanceMode.ENFORCE,
    policies=[
        GovernancePolicy.tool_allowlist(["search_database", "send_email"]),
        GovernancePolicy.pii_block(["ssn", "credit_card", "email"]),
        GovernancePolicy.cost_limit(max_per_session=0.50),
        GovernancePolicy.secret_detection(),
    ],
)

print(f"Mode: {engine.mode.value}")
print(f"Policies: {[p.type for p in engine.policies]}")

## Governed Tool Execution

This function wraps tool execution with governance. Before running any tool, it evaluates all policies against the tool name and arguments.

In [ ]:
def execute_tool_with_governance(tool_name: str, tool_input: dict) -> dict:
    """Execute a tool call with governance guardrails."""
    decision = engine.evaluate(
        tool_name=tool_name,
        tool_args=json.dumps(tool_input),
    )

    print(f"  [{decision.action}] {tool_name} | "
          f"reason={decision.reason_codes} | "
          f"risk={decision.risk_score} | "
          f"{decision.evaluation_time_ms:.2f}ms")

    if decision.action == "DENY":
        return {
            "error": f"Governance blocked: {', '.join(decision.reason_codes)}",
            "decision_id": decision.decision_id,
        }

    # Simulate tool execution
    results = {
        "search_database": lambda i: {"results": f"Found records for: {i['query']}"},
        "send_email": lambda i: {"status": f"Email sent to {i['to']}"},
        "delete_records": lambda i: {"status": f"Deleted from {i['table']}"},
    }
    return results.get(tool_name, lambda i: {"error": "Unknown tool"})(tool_input)

## Example 1: Clean Tool Call (Allowed)

In [ ]:
print("--- Clean search (should ALLOW) ---")
result = execute_tool_with_governance("search_database", {"query": "Q3 revenue numbers"})
print(f"  Result: {result}")

## Example 2: PII in Tool Arguments (Blocked)

In [ ]:
print("--- Search with SSN (should DENY) ---")
result = execute_tool_with_governance("search_database", {"query": "Find records for SSN 123-45-6789"})
print(f"  Result: {result}")

## Example 3: Unauthorized Tool (Blocked)

In [ ]:
print("--- Unauthorized tool (should DENY) ---")
result = execute_tool_with_governance("delete_records", {"table": "users", "condition": "active=false"})
print(f"  Result: {result}")

## Full Agent Loop with Claude

Now let's wire governance into a real Claude tool-use conversation.

In [ ]:
def run_governed_agent(user_message: str):
    """Run a Claude agent with governance guardrails on tool calls."""
    messages = [{"role": "user", "content": user_message}]

    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )

    # Process tool calls through governance
    while response.stop_reason == "tool_use":
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\nClaude wants to call: {block.name}({block.input})")
                result = execute_tool_with_governance(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })

        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})

        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

    # Final text response
    final_text = next((b.text for b in response.content if hasattr(b, "text")), "")
    print(f"\nClaude: {final_text}")
    return final_text

In [ ]:
# This should work — clean query, authorized tool
run_governed_agent("Search our database for Q3 2025 revenue numbers")

In [ ]:
# This should be blocked — PII in tool arguments
run_governed_agent("Look up the employee with SSN 123-45-6789")

## Audit Trail

Every governance decision is recorded as a structured receipt — useful for SOC2, HIPAA, and EU AI Act compliance.

In [ ]:
print("=== Governance Audit Trail ===")
print(f"Total decisions: {len(engine.decisions)}")
print(f"Denials: {sum(1 for d in engine.decisions if d.action == 'DENY')}")
print(f"Session cost: ${engine.cumulative_cost:.4f}")
print()
for i, d in enumerate(engine.decisions):
    print(f"  [{i+1}] {d.action} | tool={d.tool_name} | reason={d.reason_codes} | risk={d.risk_score}")

## Governance Modes

| Mode | Behavior |
|------|----------|
| **ENFORCE** | Evaluates policies, blocks violations |
| **MONITOR** | Evaluates policies, records decisions, allows all through (dry run) |
| **OBSERVE** | Skips evaluation, passes through with minimal audit |

Start with MONITOR in staging, switch to ENFORCE in production.

## How This Differs from Claude's Built-in Moderation

Claude has [content moderation](https://github.com/anthropics/anthropic-cookbook/blob/main/misc/building_moderation_filter.ipynb) capabilities, but those use the LLM itself to judge content — which is probabilistic and adds latency.

This approach is complementary:
- **Deterministic** — same input + same policy = same decision, every time
- **Fast** — regex evaluation in under 2ms (no additional API call)
- **Auditable** — structured evidence records for every decision
- **Tool-level** — operates at the tool-call boundary, not the content level

Use both together: TealTiger for deterministic tool governance, Claude moderation for nuanced content judgment.

## Resources

- [TealTiger GitHub](https://github.com/agentguard-ai/tealtiger) (Apache 2.0)
- [TealTiger Docs](https://docs.tealtiger.ai)
- [PyPI](https://pypi.org/project/tealtiger/)
- [Claude Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)